# 01 - Exploratory Data Analysis
### AI-Based Detection & Classification of Industrial Fires and Persistent Thermal Sources

This notebook loads the raw, categorized + satellite-enriched FIRMS CSVs for India and the US, combines them, and does an initial exploration of the class balance and label quality before any cleaning or feature engineering happens.

**Pipeline position:** `01_eda` -> 02_preprocessing -> 03_feature_engineering -> 04_model_training -> 05_evaluation

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 1: Load & combine both countries

In [2]:
india = pd.read_csv("firms_india_50k_satellite.csv")
us = pd.read_csv("firms_us_50k_satellite.csv")
us["country"] = "US"
india["country"] = "India"

df = pd.concat([us, india], ignore_index=True)
print(f"Combined shape: {df.shape}")
df["category"].value_counts()

Combined shape: (100000, 46)


category
wildfire                      34988
agricultural                  24064
mining                        15686
industrial                    13006
unlabeled                      7131
flare                          4340
offshore_flare_or_platform      785
Name: count, dtype: int64

In [3]:
# Drop 'unlabeled' rows from training (not a real class — insufficient evidence)
unlabeled = df[df["category"] == "unlabeled"].copy()
df = df[df["category"] != "unlabeled"].copy()
print(f"Dropped {len(unlabeled)} 'unlabeled' rows from training (kept for later inference)")
print(f"Training pool: {len(df)} rows across {df['category'].nunique()} classes")

Dropped 7131 'unlabeled' rows from training (kept for later inference)
Training pool: 92869 rows across 6 classes


### Save output for the next notebook (preprocessing)

In [4]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
df.to_csv("eda_output_combined.csv", index=False)
print(f"Saved {len(df)} rows to eda_output_combined.csv for the preprocessing notebook.")

Saved 92869 rows to eda_output_combined.csv for the preprocessing notebook.
